In [1]:
import piplite
await piplite.install("openpyxl")
from openpyxl import load_workbook
import pandas as pd
import numpy as np
import math

# 假設我們已經有一個包含球員數據的Excel檔
df = pd.read_excel('NBA_allStar.xlsx', sheet_name = 'F3')
PG = df[df['Positions'] == 1]
PG = PG.reset_index(drop = True)
normal = {'Player' : PG['Player']}
Similarity = {'Player' : PG['Player']}

print(PG['Player'])
#print(df.T)

for i in PG.columns[2:len(PG.columns)]:
    normal[i] = []
    for j in PG[i]:
        n = j/max(PG[i])
        normal[i].append(n)

df_normal = pd.DataFrame(normal)
#print(df_normal)

'''
stand = []
for i in df_normal.columns[1:len(df_normal.columns)]:
    stand.append(round(df_normal[i].std(), 2))
'''

distribute = [15, 15, 15, 6, 6, 5, 5, 5, 2, 5, 2, 5, 2, 3, 2, 7]
divisor =  sum(distribute)
#print(divisor)
weight = [x / divisor for x in distribute]
#print(weight)

0                 Kyle Lowry
1                  John Wall
2              Isaiah Thomas
3              Stephen Curry
4          Russell Westbrook
5                 Chris Paul
6               Kyrie Irving
7               Kemba Walker
8             Damian Lillard
9                Ben Simmons
10          D'Angelo Russell
11               Mike Conley
12              Jrue Holiday
13    Shai Gileous-Alexander
14           Dejounte Murray
15                Trae Young
16               Luka Doncic
17             Fred VanVleet
18              De'Aaron Fox
19             Jalen Brunson
Name: Player, dtype: object


In [19]:
k = 16
# MSE
Similarity['MSE'] = []

for i in df_normal.T:
    Similarity['MSE'].append(sum(((df_normal.T[i]['PTS':] - df_normal.T[k]['PTS':])**2)*weight))

# PSNR
Similarity['PSNR'] = []

for i in df_normal.T:
    if(Similarity['MSE'][i] != 0):
        Similarity['PSNR'].append(20*(math.log10(1 / Similarity['MSE'][i])**(1/2)))
    else:
        Similarity['PSNR'].append(33.710500)

#UQI
Similarity['UQI'] = []

X_av = sum(df_normal.T[k]['PTS':]*weight)
X_var = sum((df_normal.T[k]['PTS':] - X_av)**2)

for i in df_normal.T:
    Y_av = sum(df_normal.T[i]['PTS':]*weight)
    Y_var = sum((df_normal.T[i]['PTS':] - Y_av)**2)
    X_Y_var = sum((df_normal.T[k]['PTS':] - X_av)*(df_normal.T[i]['PTS':] - Y_av))
    Q = 4*(X_Y_var*X_av*Y_av)/((X_var + Y_var)*(X_av**2 + Y_av**2))
    Similarity['UQI'].append(Q)

In [20]:
from scipy.ndimage import uniform_filter, gaussian_filter
from scipy import signal 

def filter2(img, fltr, mode = 'same'):
    return signal.convolve2d(img, np.rot90(fltr, 2), mode = mode)

def fspecial(fltr, ws, **kwargs):
    if fltr == uniform_filter:
        return np.ones((ws, ws)) / ws**2
    elif fltr == gaussian_filter:
        x, y = np.mgrid[-ws//2 + 1:ws//2 + 1, -ws//2 + 1:ws//2 + 1]
        g = np.exp(-((x**2 + y**2)/(2.0*kwargs['sigma']**2)))
        g[g < np.finfo(g.dtype).eps*g.max()] = 0
        assert g.shape == (ws, ws)
        den = g.sum()
        if den != 0:
            g /= den
        return g
    return None

def _get_sums(GT, P, win, mode = 'same'):
    mu1, mu2 = (filter2(GT, win, mode), filter2(P, win, mode))
    return mu1*mu1, mu2*mu2, mu1*mu2

def _get_sigmas(GT, P, win, mode = 'same', **kwargs):
    if 'sums' in kwargs:
        GT_sum_sq, P_sum_sq, GT_P_sum_mul = kwargs['sums']
    else:
        GT_sum_sq, P_sum_sq, GT_P_sum_mul = _get_sums(GT, P, win, mode)

    return filter2(GT*GT, win, mode) - GT_sum_sq, filter2(P*P, win, mode) - P_sum_sq, filter2(GT*P, win, mode) - GT_P_sum_mul

def _vifp_value(X, Y, sigma_nsq):
    EPS = 1e-10
    num =0.0
    den =0.0
    GT = np.array([X])
    P = np.array([Y])
    
    for scale in range(1,5):
        N = 2.0**(4 - scale + 1) + 1
        win = fspecial(gaussian_filter, ws = N, sigma = N/5)
    
        if scale > 1:
            GT = filter2(GT,win,'valid')[::2, ::2]
            #print('GT :', GT)
            P = filter2(P,win,'valid')[::2, ::2]
            #print('P :', P)
    
        GT_sum_sq, P_sum_sq, GT_P_sum_mul = _get_sums(GT, P, win, mode = 'valid')
        sigmaGT_sq, sigmaP_sq, sigmaGT_P = _get_sigmas(GT, P, win, mode = 'valid', sums = (GT_sum_sq, P_sum_sq, GT_P_sum_mul))
        
        sigmaGT_sq[sigmaGT_sq < 0] = 0
        sigmaP_sq[sigmaP_sq < 0] = 0
        
        g = sigmaGT_P / (sigmaGT_sq + EPS)
        sv_sq = sigmaP_sq - g*sigmaGT_P
    	
        g[sigmaGT_sq < EPS] = 0
        sv_sq[sigmaGT_sq < EPS] = sigmaP_sq[sigmaGT_sq < EPS]
        sigmaGT_sq[sigmaGT_sq < EPS] = 0
    	
        g[sigmaP_sq < EPS] = 0
        sv_sq[sigmaP_sq<EPS] = 0
        
        sv_sq[g < 0] = sigmaP_sq[g < 0]
        g[g < 0] = 0
        sv_sq[sv_sq <= EPS] = EPS
        
        num += np.sum(np.log2(1.0 + (g**2.)*sigmaGT_sq / (sv_sq + sigma_nsq)))
        den += np.sum(np.log2(1.0 + sigmaGT_sq / sigma_nsq))
        
    return num/den

In [21]:
import statistics as st

Year = ['Second', 'Third']
df = pd.read_excel('NBA_allStar.xlsx', sheet_name = 'First')
PG = df[df['Positions'] == 1] # Need Fixed
PG = PG.reset_index(drop = True)
merge = {'Player' : PG['Player']}

for box in PG.columns[2:]:
    merge[box] = []
    for i in PG[box]:
        merge[box].append([i])
        
for year in Year:
    df = pd.read_excel('NBA_allStar.xlsx', sheet_name = year)
    PG = df[df['Positions'] == 1] # Need Fixed
    PG = PG.reset_index(drop = True)

    for box in PG.columns[2:]:
        for i in range(len(PG[box])):
            merge[box][i].append(PG[box][i])

F3 = pd.DataFrame(merge)
#print(F3)

In [22]:
#VIF
Similarity['VIF_SE'] = []
for i in range(len(F3['Player'])):
    data = pd.DataFrame({
    'Seasons': [1, 2, 3],
    'Player_A_Score': F3.T[k]['PTS'],
    'Player_A_Rebounds': F3.T[k]['REB'],
    'Player_A_Assists': F3.T[k]['AST'],
    'Player_A_Steals': F3.T[k]['STL'],
    'Player_A_Blocks': F3.T[k]['BLK'],
    'Player_A_Turnovers': F3.T[k]['TOV'],
    'Player_A_Fouls': F3.T[k]['PF'],
    'Player_A_FGP': F3.T[k]['FG%'],
    'Player_A_FGA': F3.T[k]['FGA'],
    'Player_A_2PP': F3.T[k]['2P%'],
    'Player_A_2PA': F3.T[k]['2PA'],
    'Player_A_3PP': F3.T[k]['3P%'],
    'Player_A_3PA': F3.T[k]['3PA'],
    'Player_A_FTP': F3.T[k]['FT%'],
    'Player_A_FTA': F3.T[k]['FTA'],
    'Player_A_USGP': F3.T[k]['USG'],
    'Player_B_Score': F3.T[i]['PTS'],
    'Player_B_Rebounds': F3.T[i]['REB'],
    'Player_B_Assists': F3.T[i]['AST'],
    'Player_B_Steals': F3.T[i]['STL'],
    'Player_B_Blocks': F3.T[i]['BLK'],
    'Player_B_Turnovers': F3.T[i]['TOV'],
    'Player_B_Fouls': F3.T[i]['PF'],
    'Player_B_FGP': F3.T[i]['FG%'],
    'Player_B_FGA': F3.T[i]['FGA'],
    'Player_B_2PP': F3.T[i]['2P%'],
    'Player_B_2PA': F3.T[i]['2PA'],
    'Player_B_3PP': F3.T[i]['3P%'],
    'Player_B_3PA': F3.T[i]['3PA'],
    'Player_B_FTP': F3.T[i]['FT%'],
    'Player_B_FTA': F3.T[i]['FTA'],
    'Player_B_USGP': F3.T[i]['USG'],
    })

    features = ['Score', 'Rebounds', 'Assists', 'Steals', 'Blocks', 'Turnovers', 'Fouls',
                'FGP', 'FGA', '2PP', '2PA', '3PP', '3PA', 'FTP', 'FTA', 'USGP']
    VIF_AtoB = {}
    VIF_BtoA = {}
    VIF = {}
    SE_VIF = 0
    
    for feature in features:
        VIF_AtoB[feature] = _vifp_value(data[f'Player_A_{feature}'], data[f'Player_B_{feature}'], 2)
        VIF_BtoA[feature] = _vifp_value(data[f'Player_B_{feature}'], data[f'Player_A_{feature}'], 2)
        VIF[feature] = (VIF_AtoB[feature] + VIF_BtoA[feature])/2
        SE_VIF = SE_VIF + (VIF[feature] - 1)**2

    if SE_VIF < 1e-10:
        SE_VIF = 0

    #print(PG['Player'][i])
    #print("VIF 指標（每項指標）：")
    #for feature in features:
        #print(f"{feature}: {VIF[feature]:.4f}")

    #print(sums)
    Similarity['VIF_SE'].append(SE_VIF)


df_Similarity= pd.DataFrame(Similarity)

print('For', PG['Player'][k], "'s similarity:")
print(df_Similarity)

For Luka Doncic 's similarity:
                    Player       MSE       PSNR       UQI        VIF_SE
0               Kyle Lowry  0.149629  18.165720 -0.303882  3.723560e+01
1                John Wall  0.100781  19.966167 -0.313564  2.701591e+02
2            Isaiah Thomas  0.121658  19.129642  0.236765  3.063807e+01
3            Stephen Curry  0.105361  19.771892 -0.194101  9.231546e+00
4        Russell Westbrook  0.071523  21.406125  0.070733  1.528601e+02
5               Chris Paul  0.112198  19.493751 -0.044632  6.325428e+01
6             Kyrie Irving  0.079205  20.988082  0.044276  2.031209e+00
7             Kemba Walker  0.103074  19.868072 -0.037337  2.731335e+00
8           Damian Lillard  0.098261  20.076027  0.401720  2.126937e+00
9              Ben Simmons  0.098279  20.075249 -0.218892  8.259089e+06
10        D'Angelo Russell  0.081372  20.876082  0.203599  1.762174e+00
11             Mike Conley  0.172209  17.480770 -0.061713  2.967767e+01
12            Jrue Holiday  0.143

In [ ]:
'''
top_5_MSE = df_Similarity['MSE'].nsmallest(6)

print('MSE:')
for i in range(1, 6):
    print(PG['Player'][top_5_MSE.index[i]])
    print(top_5_MSE.values[i])
'''

In [ ]:
'''
top_5_PSNR = df_Similarity['PSNR'].nlargest(6)

print('PSNR:')
for i in range(1, 6):
    print(PG['Player'][top_5_PSNR.index[i]])
    print(top_5_PSNR.values[i])
'''

In [ ]:
'''
top_5_UQI = df_Similarity['UQI'].nlargest(6)

print('UQI:')
for i in range(1, 6):
    print(PG['Player'][top_5_UQI.index[i]])
    print(top_5_UQI.values[i])
'''

In [ ]:
'''
top_5_VIF_SE = df_Similarity['VIF_SE'].nsmallest(6)

print('VIF_SE:')
for i in range(1, 6):
    print(PG['Player'][top_5_VIF_SE.index[i]])
    print(top_5_VIF_SE.values[i])
'''

In [23]:
#standard
#For MSE
MSE_a = sum(((df_normal.T[k]['PTS':] - 0.8*df_normal.T[k]['PTS':])**2)*weight)
MSE_b = sum(((df_normal.T[k]['PTS':] - 0.6*df_normal.T[k]['PTS':])**2)*weight)
MSE_c = sum(((df_normal.T[k]['PTS':] - 0.4*df_normal.T[k]['PTS':])**2)*weight)
MSE_d = sum(((df_normal.T[k]['PTS':] - 0.2*df_normal.T[k]['PTS':])**2)*weight)

print(MSE_a, MSE_b, MSE_c, MSE_d)

#For PSNR
PSNR_a = 20*(math.log10(1 / MSE_a)**(1/2))
PSNR_b = 20*(math.log10(1 / MSE_b)**(1/2))
PSNR_c = 20*(math.log10(1 / MSE_c)**(1/2))
PSNR_d = 20*(math.log10(1 / MSE_d)**(1/2))

print(PSNR_a, PSNR_b, PSNR_c, PSNR_d)

#For UQI
Y_av = sum(0.8*df_normal.T[k]['PTS':]*weight)
Y_var = sum((0.8*df_normal.T[k]['PTS':] - Y_av)**2)
X_Y_var = sum((df_normal.T[k]['PTS':] - X_av)*(0.8*df_normal.T[k]['PTS':] - Y_av))
Q_a = 4*(X_Y_var*X_av*Y_av)/((X_var + Y_var)*(X_av**2 + Y_av**2))

Y_av = sum(0.6*df_normal.T[k]['PTS':]*weight)
Y_var = sum((0.6*df_normal.T[k]['PTS':] - Y_av)**2)
X_Y_var = sum((df_normal.T[k]['PTS':] - X_av)*(0.6*df_normal.T[k]['PTS':] - Y_av))
Q_b = 4*(X_Y_var*X_av*Y_av)/((X_var + Y_var)*(X_av**2 + Y_av**2))

Y_av = sum(0.4*df_normal.T[k]['PTS':]*weight)
Y_var = sum((0.4*df_normal.T[k]['PTS':] - Y_av)**2)
X_Y_var = sum((df_normal.T[k]['PTS':] - X_av)*(0.4*df_normal.T[k]['PTS':] - Y_av))
Q_c = 4*(X_Y_var*X_av*Y_av)/((X_var + Y_var)*(X_av**2 + Y_av**2))

Y_av = sum(0.2*df_normal.T[k]['PTS':]*weight)
Y_var = sum((0.2*df_normal.T[k]['PTS':] - Y_av)**2)
X_Y_var = sum((df_normal.T[k]['PTS':] - X_av)*(0.2*df_normal.T[k]['PTS':] - Y_av))
Q_d = 4*(X_Y_var*X_av*Y_av)/((X_var + Y_var)*(X_av**2 + Y_av**2))

print(Q_a, Q_b, Q_c, Q_d)

#For VIF
SE_VIF_a = 0
SE_VIF_b = 0
SE_VIF_c = 0
SE_VIF_d = 0

for feature in features:
        temp_a = _vifp_value(data[f'Player_A_{feature}'], 0.8*data[f'Player_A_{feature}'], 2)
        SE_VIF_a = SE_VIF_a + (temp_a - 1)**2
        temp_b = _vifp_value(data[f'Player_A_{feature}'], 0.6*data[f'Player_A_{feature}'], 2)
        SE_VIF_b = SE_VIF_b + (temp_b - 1)**2
        temp_c = _vifp_value(data[f'Player_A_{feature}'], 0.4*data[f'Player_A_{feature}'], 2)
        SE_VIF_c = SE_VIF_c + (temp_c - 1)**2
        temp_d = _vifp_value(data[f'Player_A_{feature}'], 0.2*data[f'Player_A_{feature}'], 2)
        SE_VIF_d = SE_VIF_d + (temp_d - 1)**2

print(SE_VIF_a, SE_VIF_b, SE_VIF_c, SE_VIF_d)

0.0314244648326196 0.1256978593304785 0.2828201834935765 0.5027914373219139
24.517194861850747 18.980749388861746 14.812016747162776 10.929082799191823
0.9518143961927423 0.778546712802768 0.47562425683709875 0.14792899408284027
1.601453143766208 5.4528579347506385 10.150102104491413 14.237772888068545


In [25]:
from scipy.linalg import lstsq

# 假設你有以下 5 個方程式和 4 個未知數 (x, y, z, w)
# 例子數據 (把係數和常數項替換為實際數值)
A = np.array([
    [MSE_a, PSNR_a, Q_a, SE_VIF_a],  # 第1個方程式的係數
    [MSE_b, PSNR_b, Q_b, SE_VIF_b],  # 第2個方程式的係數
    [MSE_c, PSNR_c, Q_c, SE_VIF_c],  # 第3個方程式的係數
    [MSE_d, PSNR_d, Q_d, SE_VIF_d],  # 第4個方程式的係數
    [0, 33.710500, 1, 0]   # 第5個方程式的係數
])

# 常數項
B = np.array([80, 60, 40, 20, 100])

# 使用最小二乘法求解 Ax = B
solution, residuals, rank, s = lstsq(A, B)

# 顯示解
x, y, z, w = solution
print(f"x = {x}, y = {y}, z = {z}, w = {w}")
print(f"Residuals (誤差平方和): {residuals}")

x = 15.84923866849718, y = 1.9256950830167086, z = 35.189693943619936, w = -0.993030367674574
Residuals (誤差平方和): 0.5556696962250978


In [425]:
print('For', df_normal.T[k]['Player'], "'s similaities:")
result_2 = []

for i in df_normal.T:
    temp = df_Similarity.T[i]['MSE']*x + df_Similarity.T[i]['PSNR']*y +\
             df_Similarity.T[i]['UQI']*z + df_Similarity.T[i]['VIF_SE']*w
    result_2.append(temp)
    
    print(df_normal.T[i]['Player'], ':', temp)

For Trae Young 's similaities:
Kyle Lowry : 13.950735822667273
John Wall : -131.4790935771622
Isaiah Thomas : 36.633343741142305
Stephen Curry : 41.471966175901755
Russell Westbrook : -26.271245907652954
Chris Paul : 46.76402916460415
Kyrie Irving : 58.77060383626707
Kemba Walker : 51.756220420369374
Damian Lillard : 67.51799671716121
Ben Simmons : -4855994.971167907
D'Angelo Russell : 57.278583224891925
Mike Conley : 39.85642373660556
Jrue Holiday : 18.34023185510194
Shai Gileous-Alexander : -4.413642348168999
Dejounte Murray : -0.698392214049278
Trae Young : 100.08931058859397
Luka Doncic : 63.20719618389194
Fred VanVleet : 24.09269091672345
De'Aaron Fox : 41.06617390436135
Jalen Brunson : 46.36504792846249


In [426]:
out = pd.DataFrame({'Player': PG['Player'], 'Similarity': result_2})

# 2. 輸出成 Excel（需已安裝 openpyxl）
out.to_excel('similarity_scores.xlsx', index=False, sheet_name='Similarity')

In [428]:
import heapq
top_5_similarity = heapq.nlargest(5, enumerate(result_2[0:13]), key = lambda x:x[1]) # Need Fixed
top_5_indices = [index for index, value in top_5_similarity]
top_5_values = [value for index, value in top_5_similarity]
print('The top 5 similarities:')
for i in range(5):
    print(df_normal.T[top_5_indices[i]]['Player'], ':', top_5_values[i])


The top 5 similarities:
Damian Lillard : 67.51799671716121
Kyrie Irving : 58.77060383626707
D'Angelo Russell : 57.278583224891925
Kemba Walker : 51.756220420369374
Chris Paul : 46.76402916460415


In [ ]:
'''
#Prediction for the performance after 3 years.
Prediction_A3= {'Player' : PG['Player'][k]}

for i in PG.columns[2:len(PG.columns)]:
    all_prediction = PG[i][k] * 600
    all_value = 600

    for j in range(0, 6):
        all_prediction = all_prediction + PG[i][top_5_indices[j]] * top_5_values[j] * (6 - j)
        all_value = all_value + top_5_values[j] * (6 - j)
        
    prediction = all_prediction / all_value
    Prediction_A3[i] = prediction
    if i in ['3PA', '2PA']:
        Prediction_A3[i] = prediction * 1.22
    elif i in ['FTA']:
        Prediction_A3[i] = prediction * 1.07
    elif i in ['USG%']:
        Prediction_A3[i] = prediction + 5
    else:
        Prediction_A3[i] = prediction
print(Prediction_A3['PTS'], ',', Prediction_A3['FGA'])
Prediction_A3['FGA'] = Prediction_A3['2PA'] + Prediction_A3['3PA']
Prediction_A3['PTS'] = 2*Prediction_A3['2PA']*Prediction_A3['2P%'] + 3*Prediction_A3['3PA']*Prediction_A3['3P%'] + Prediction_A3['FTA']*Prediction_A3['FT%']

print('The prediction of future:')
for key, value in Prediction_A3.items():
    print(f"{key}: {value}")
'''

In [429]:
df_A3 = pd.read_excel('NBA_allStar.xlsx', sheet_name = 'A3')
PG_A3 = df_A3[df_A3['Positions'] == 1] #Need fix
PG_A3 = PG_A3.reset_index(drop = True)

#print(PG_A3)

In [430]:
Prediction_A32= {'Player' : PG_A3['Player'][k]}

for i in PG_A3.columns[2:len(PG_A3.columns)]:
    all_prediction = PG[i][k] * 600
    all_value = 600

    for j in range(0, 5):
        all_prediction = all_prediction + PG_A3[i][top_5_indices[j]] * top_5_values[j] * (5 - j)
        all_value = all_value + top_5_values[j] * (5 - j)
        
    prediction = all_prediction / all_value
    Prediction_A32[i] = prediction
    
    if i in ['2PA']:
        Prediction_A32[i] = prediction  / 1.05
    elif i in ['3PA']:
        Prediction_A32[i] = prediction * 1.3
    elif i in ['USG']:
        Prediction_A32[i] = prediction + 2
    else:
        Prediction_A32[i] = prediction
#print(Prediction_A32['PTS'], ',', Prediction_A32['FGA'])
Prediction_A32['FGA'] = Prediction_A32['2PA'] + Prediction_A32['3PA']
Prediction_A32['PTS'] = 2*Prediction_A32['2PA']*Prediction_A32['2P%'] + 3*Prediction_A32['3PA']*Prediction_A32['3P%'] + Prediction_A32['FTA']*Prediction_A32['FT%']

print('The prediction of future:')
for key, value in Prediction_A32.items():
    if key in ['Player']:
        print(f"{key}: {value}")
    elif key in ['FG%', '2P%', '3P%', 'FT%']:
        value = int(value * 1000)/1000
        Prediction_A32[key] = value
        print(f"{key}: {value}")
    else:
        value = int(value * 10)/10
        Prediction_A32[key] = value
        print(f"{key}: {value}")

The prediction of future:
Player: Trae Young
PTS: 27.0
REB: 4.1
AST: 7.9
STL: 1.1
BLK: 0.2
TOV: 3.4
PF: 1.9
FG%: 0.44
FGA: 20.4
2P%: 0.486
2PA: 11.3
3P%: 0.362
3PA: 9.1
FT%: 0.874
FTA: 6.9
USG: 31.4


In [431]:
#MSE for prediction
#sum_A3 = []
sum_A32 = []

for i in PG.columns[2:len(PG.columns)]:
    #sum_A3.append((Prediction_A3[i] - PG_A3[i][k])**2)
    sum_A32.append((Prediction_A32[i] - PG_A3[i][k])**2)

#MSE_A3 = 0
MSE_A32 = 0
for j in range(len(weight)):
    #MSE_A3 = MSE_A3 + sum_A3[j] * weight[j]
    MSE_A32 = MSE_A32 + sum_A32[j] * weight[j]

print('MSE for prediction:')
#print(MSE_A3)
print(MSE_A32)

MSE for prediction:
1.3043049000000002


In [432]:
for key, value in Prediction_A32.items():
    print(f"{key}: {value}", ',', PG_A3[key][k])

Player: Trae Young , Trae Young
PTS: 27.0 , 27.5
REB: 4.1 , 3.3
AST: 7.9 , 10.4
STL: 1.1 , 1.1
BLK: 0.2 , 0.1
TOV: 3.4 , 4.2
PF: 1.9 , 1.7
FG%: 0.44 , 0.442
FGA: 20.4 , 19.9
2P%: 0.486 , 0.491
2PA: 11.3 , 12.1
3P%: 0.362 , 0.365
3PA: 9.1 , 7.8
FT%: 0.874 , 0.884
FTA: 6.9 , 8.1
USG: 31.4 , 32.7


In [433]:
for i in PG.columns[2:len(PG.columns)]:
    if PG_A3[i][k] == 0:
        e = abs(Prediction_A32[i])
    else:
        e = int(abs(Prediction_A32[i] - PG_A3[i][k]) / PG_A3[i][k]*1000)/1000
    print(i, ':', f"{e}")

PTS : 0.018
REB : 0.242
AST : 0.24
STL : 0.0
BLK : 1.0
TOV : 0.19
PF : 0.117
FG% : 0.004
FGA : 0.025
2P% : 0.01
2PA : 0.066
3P% : 0.008
3PA : 0.166
FT% : 0.011
FTA : 0.148
USG : 0.039


In [434]:
sum_diff = 0
sum_len = 0

for i in PG.columns[2:len(PG.columns)]:
    sum_diff = sum_diff + (Prediction_A32[i]-PG_A3[i][k])**2
    sum_len = sum_len + (PG_A3[i][k])**2

E = (sum_diff**(1/2))/(sum_len**(1/2))

print(E*100)

7.166261027879693


In [435]:
FP_predict = Prediction_A32['PTS'] + 1.2 * Prediction_A32['REB'] + 1.5 * Prediction_A32['AST'] + 3 * Prediction_A32['STL'] + 3 * Prediction_A32['BLK'] - Prediction_A32['TOV']
FP_A3 = PG_A3['PTS'][k] + 1.2 * PG_A3['REB'][k] + 1.5 * PG_A3['AST'][k] + 3 * PG_A3['STL'][k] + 3 * PG_A3['BLK'][k] - PG_A3['TOV'][k]
MAPE = 100 * int(abs((FP_predict - FP_A3)/FP_A3) * 10000)/10000

print(MAPE)

4.71
